# 02 Merge the Earth Engine exports into the features table

Merges the per-layer CSVs that `gee/export_frantz_features.js` writes to Google Drive (18 layers x 104 features, training
and natural-test sets) into one table with `split` = train / test / excluded / unused.

**Inputs:** download the Drive folders `frantz_features` and `frantz_features_natural` into `gee_exports/`; the file
`data/unused_existing_ids.csv` (shipped) marks the training rows the rebalancing left out.
**Output:** `data/features_full_1872_split.csv`. This is the table already provided by `scripts/download_data.py`, so you
only need this notebook if you re-extracted the features yourself. Earth Engine data change over time (reprocessing,
new scenes), so a fresh extraction will not be bit-identical to the shipped table.

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

In [ ]:
#!/usr/bin/env python3
"""
merge_gee_exports.py

Merges the per-layer CSVs exported by export_frantz_features_gee (js or py)
(<city>_<layer>.csv, 104 features each) into one table:

    city, sample_id, height_m, is_background, x_m, y_m, <1872 features>

Only these metadata columns are kept. Columns the sampler writes that are
derived from the label (slice_lo, n_parts_50m) are dropped on purpose so
they can never be used as predictors. Feature selection is NOT done here:
it happens inside the cross-validation folds in train_models.py.

EXTENSION (beyond Frantz et al. 2021): --extras VVH also merges
<city>_VVH.csv (8 features: VVH and its 7 morphological operators), giving
1880 features. Without --extras the table is the 1872-feature Frantz set.

TEST SET (deviation from Frantz, who validates on the stratified sample): with --test-dir the
natural-distribution test exports (same 18 layer files per city, ids <city>_nat_NNNNNN) are merged
with the same code and appended to the same table, with a `split` column:
    train     the stratified training/validation points
    test      the natural test points (touch only for the final evaluation)
    excluded  training points flagged by the 50 m rule (--drop-train lists them); keep them out
              of training, tuning and CV
`split` is also a metadata column, so it can never be picked up as a predictor by code that
takes "every column not in META".

--drop-radius 50 recomputes the exclusion from the coordinates in the table itself (a training row
is `excluded` when a test row of the same city lies within that many metres, Euclidean, strictly
less than), which is the rule natural_test_points(drop_train=True, buffer_m=50) used. It needs no
drop-list files, so it works from any folder. With --drop-train as well, the two are compared. Without --test-dir or --drop-train the output has no
`split` column and is identical to before.

Usage (terminal):
    python merge_gee_exports.py --dir gee_exports/frantz_features \
        --cities porto_alegre,sao_paulo,rio_de_janeiro --out s1s2_dataset/features_full_1872.csv
    python merge_gee_exports.py --extras VVH --out s1s2_dataset/features_full_1880.csv
    python merge_gee_exports.py --dir gee_exports/frantz_features \
        --test-dir gee_exports/frantz_features_natural \
        --drop-train data/porto_alegre/sample_points_natural_drop_train.csv \
                     data/sao_paulo/sample_points_natural_drop_train.csv \
                     data/rio_de_janeiro/sample_points_natural_drop_train.csv
    main(["--selftest"])

In a notebook, call main([...]) with the same arguments as a list, e.g.
    main(["--cities", "porto_alegre,sao_paulo,rio_de_janeiro", "--extras", "VVH"])
"""

import argparse
import os
import sys
import tempfile

import numpy as np
import pandas as pd

S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
INDICES = ["NDVI", "TCG", "NDBI", "TCB", "TCW", "mNDWI"]
BASE_LAYERS = ["VV", "VH"] + S2_BANDS + INDICES
EXT_LAYERS = ["VVH"]
STATS = ["mean", "stdDev", "p0", "p10", "p25", "p50", "p75", "p90", "p100",
         "skew", "kurtosis", "range", "IQR"]
OPS = ["ERO", "DIL", "OPN", "CLS", "GRD", "THT", "BHT"]
META = ["sample_id", "city", "height_m", "is_background", "x_m", "y_m"]
SPLITS = ["train", "test", "excluded"]


def expected_features(layer):
    if layer == "VVH":                      # single static layer, no STMs
        return ["VVH"] + [f"VVH_{op}" for op in OPS]
    stm = [f"{layer}_{s}" for s in STATS]
    return stm + [f"{n}_{op}" for n in stm for op in OPS]


def merge_city(city, directory, layers):
    merged, problems = None, []
    for layer in layers:
        path = os.path.join(directory, f"{city}_{layer}.csv")
        if not os.path.exists(path):
            problems.append(f"missing file {path}")
            continue
        df = pd.read_csv(path)
        missing = [c for c in expected_features(layer) if c not in df.columns]
        if missing:
            problems.append(f"{path}: {len(missing)} expected columns absent (e.g. {missing[:2]})")
            continue
        keep = expected_features(layer)
        if merged is None:
            merged = df[META + keep].copy()
        else:
            merged = merged.merge(df[["sample_id"] + keep], on="sample_id", how="inner")
            if len(merged) != len(df):
                problems.append(f"{path}: row count changed after merge ({len(df)} -> {len(merged)})")
    return merged, problems


def _parse(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dir", default=config.GEE_FEATURES)
    ap.add_argument("--cities", default="porto_alegre,sao_paulo,rio_de_janeiro,recife,salvador,curitiba,niteroi,belo_horizonte")
    ap.add_argument("--out", default=None, help="default: data/features_full_<n>[_split].csv")
    ap.add_argument("--extras", default="", help="comma list of extension layers, e.g. VVH")
    ap.add_argument("--test-dir", default=config.GEE_FEATURES_NATURAL,
                    help="folder with the natural test exports; adds split=test rows. Pass \"\" for a training-only table")
    ap.add_argument("--drop-train", nargs="*", default=[],
                    help="one or more *_natural_drop_train.csv files; those training rows get split=excluded")
    ap.add_argument("--drop-radius", type=float, default=50.0,
                    help="metres; mark training rows within this distance of a test row of the same city as excluded (0 = off)")
    ap.add_argument("--unused-ids", nargs="*", default=[str(config.UNUSED_IDS)],
                    help="one or more CSVs with a sample_id column (rebalance_training_cell.py writes unused_existing_ids.csv); "
                         "those training rows get split=unused: not excluded by the 50 m rule, just left out of the balanced training set")
    ap.add_argument("--selftest", action="store_true")
    # parse_known_args so Jupyter's own -f kernel.json argument is ignored
    args, _ = ap.parse_known_args(sys.argv[1:] if argv is None else argv)
    return args


def main(argv=None):
    args = _parse(argv)
    extras = [e for e in args.extras.split(",") if e]
    bad = [e for e in extras if e not in EXT_LAYERS]
    if bad:
        raise SystemExit(f"Unknown extras {bad}. Valid: {EXT_LAYERS}")
    layers = BASE_LAYERS + extras
    n_expected = 1872 + sum(len(expected_features(e)) for e in extras)

    if args.selftest:
        d = tempfile.mkdtemp()
        rng = np.random.default_rng(0)

        def fake_exports(folder, city, n, tag, xy=None):
            os.makedirs(folder, exist_ok=True)
            ids = [f"{city}_{tag}{i:06d}" for i in range(n)]
            for layer in BASE_LAYERS + EXT_LAYERS:
                cols = expected_features(layer)
                df = pd.DataFrame(rng.normal(size=(n, len(cols))), columns=cols)
                df.insert(0, "sample_id", ids)
                df.insert(1, "city", city)
                df.insert(2, "height_m", np.linspace(0, 60, n))
                df.insert(3, "is_background", (np.arange(n) % 10 == 0).astype(int))
                df.insert(4, "x_m", np.arange(n) * 60.0 if xy is None else xy[0])
                df.insert(5, "y_m", np.arange(n) * 30.0 if xy is None else xy[1])
                df["slice_lo"], df["n_parts_50m"] = 7, 12     # must NOT reach the output
                df["split"], df["dist_to_train_m"] = "test", 80.0   # must NOT leak in from the export
                df["system:index"], df[".geo"] = range(n), "{}"
                df.sample(frac=1, random_state=1).to_csv(os.path.join(folder, f"{city}_{layer}.csv"), index=False)
            return ids

        tr, te = os.path.join(d, "train"), os.path.join(d, "test")
        drops = []
        for city, n, nt in [("porto_alegre", 120, 40), ("sao_paulo", 90, 30)]:
            ids = fake_exports(tr, city, n, "")
            txs, tys = 1e6 + np.arange(nt) * 60.0, 1e6 + np.arange(nt) * 30.0
            txs[0], tys[0] = 300.0, 160.0       # 10 m from training point 5 (300, 150); the rest are far away
            fake_exports(te, city, nt, "nat_", (txs, tys))
            dp = os.path.join(d, f"{city}_drop.csv")
            pd.DataFrame({"sample_id": ids[:3] + ["not_a_real_id"]}).to_csv(dp, index=False)
            drops.append(dp)
        args.dir, args.cities = tr, "porto_alegre,sao_paulo"
        args.test_dir, args.drop_train = te, drops
        args.out = os.path.join(d, "features.csv")
        args.unused_ids = []
        args.drop_radius = None

    use_split = bool(args.test_dir or args.drop_train or args.drop_radius)
    out = args.out or os.path.join(str(config.DATA_DIR), f"features_full_{n_expected}" + ("_split" if use_split else "") + ".csv")
    all_problems, parts = [], []
    for label, folder in [("train", args.dir)] + ([("test", args.test_dir)] if args.test_dir else []):
        frames = []
        for city in args.cities.split(","):
            merged, problems = merge_city(city, folder, layers)
            all_problems += [f"[{label}] {p}" for p in problems]
            if merged is not None:
                print(f"{label:>5} {city}: {len(merged)} samples, {merged.shape[1] - len(META)} features")
                frames.append(merged.copy().assign(split=label))
        if not frames:
            hint = ""
            if os.path.isdir(folder) and any(f.endswith(".tfrecord.gz") or f.endswith(".tfrecord") for f in os.listdir(folder)):
                hint = (" That folder holds TFRecord patch files, which this script cannot read; the table "
                        "export (PRODUCT = 'table', POINTS_SET = 'natural') writes CSVs to frantz_features_natural.")
            seen = sorted(os.listdir(folder))[:12] if os.path.isdir(folder) else "folder does not exist"
            why = "\n  ".join(q for q in all_problems if q.startswith(f"[{label}]"))[:1500]
            raise SystemExit(f"Nothing merged from {label} folder {folder}.{hint}\nFirst files found: {seen}\n"
                             f"Expected names like <city>_<layer>.csv (e.g. porto_alegre_VV.csv). Reasons:\n  {why}")
        parts.append(pd.concat(frames, ignore_index=True))

    table = pd.concat(parts, ignore_index=True)

    # the natural points are only valid if their ids and positions do not collide with training ids
    if not table.sample_id.is_unique:
        raise SystemExit("sample_id is not unique across train and test: " +
                         str(table.sample_id[table.sample_id.duplicated()].head().tolist()))

    radius_ids = set()
    if args.drop_radius:
        from scipy.spatial import cKDTree
        for city, g in table.groupby("city"):
            te_, tr_ = g[g.split == "test"], g[g.split == "train"]
            if te_.empty or tr_.empty:
                continue
            d = cKDTree(te_[["x_m", "y_m"]].to_numpy(float)).query(tr_[["x_m", "y_m"]].to_numpy(float))[0]
            ids_ = set(tr_.sample_id[d < args.drop_radius])
            radius_ids |= ids_
            print(f"drop-radius {args.drop_radius:g} m: {len(ids_)} of {len(tr_)} {city} training rows flagged")
        table.loc[table.sample_id.isin(radius_ids), "split"] = "excluded"

    for path in args.drop_train:
        ids = pd.read_csv(path, usecols=["sample_id"]).sample_id
        if args.drop_radius:        # already marked from the coordinates: compare with the sampler's list
            pref = ids.iloc[0].rsplit("_", 1)[0] if len(ids) else None
            diff = set(ids) ^ {i for i in radius_ids if i.rsplit("_", 1)[0] == pref}
            print(f"{os.path.basename(path)}: {len(ids)} ids; differences against the coordinate rule: {len(diff)}")
            if diff:
                all_problems.append(f"{path}: {len(diff)} ids differ from --drop-radius")
            continue
        hit = table.sample_id.isin(ids) & (table.split == "train")
        unknown = int((~ids.isin(table.sample_id)).sum())
        table.loc[hit, "split"] = "excluded"
        print(f"excluded {int(hit.sum())} training rows from {os.path.basename(path)}"
              + (f"  ({unknown} ids in the list are not in the table)" if unknown else ""))
        if unknown:
            all_problems.append(f"{path}: {unknown} ids not found in the merged table")

    for path in args.unused_ids:
        ids = pd.read_csv(path, usecols=["sample_id"]).sample_id
        hit = table.sample_id.isin(ids) & (table.split == "train")
        print(f"unused: {int(hit.sum())} training rows from {os.path.basename(path)} set to split=unused "
              f"({int((~ids.isin(table.sample_id)).sum())} ids not in the table)")
        table.loc[hit, "split"] = "unused"

    if all_problems:
        print("\nPROBLEMS:\n  " + "\n  ".join(all_problems))

    cols = META + (["split"] if use_split else [])
    table = table[cols + [c for c in table.columns if c not in cols and c != "split"]]
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    table.to_csv(out, index=False)
    feats = table.columns.difference(cols)
    print(f"\nWrote {out}: {len(table)} rows, {len(feats)} features, "
          f"{table[list(feats)].isna().mean().mean():.1%} missing values")
    if use_split:
        print(table.groupby(["city", "split"]).size().unstack(fill_value=0).to_string())

    if args.selftest:
        assert len(feats) == n_expected, (len(feats), n_expected)
        assert not ({"slice_lo", "n_parts_50m", "system:index", ".geo", "dist_to_train_m"} & set(table.columns))
        assert table.sample_id.is_unique and len(table) == 210 + 70
        assert list(table.columns[:7]) == META + ["split"]
        vc = table.split.value_counts().to_dict()
        assert vc == {"train": 210 - 6, "test": 70, "excluded": 6}, vc
        assert table[table.split == "test"].sample_id.str.contains("_nat_").all()
        assert table[table.split != "test"].sample_id.str.contains("_nat_").sum() == 0
        # second run: exclusion from the coordinates only
        out2 = os.path.join(d, "features_radius.csv")
        main(["--dir", tr, "--test-dir", te, "--cities", args.cities, "--drop-radius", "50", "--unused-ids", "--out", out2])
        t2 = pd.read_csv(out2, usecols=["sample_id", "split"])
        ex = sorted(t2.sample_id[t2.split == "excluded"])
        assert ex == ["porto_alegre_000005", "sao_paulo_000005"], ex
        # third run: rows the rebalancing leaves out become split=unused (never touching excluded or test rows)
        pick = t2.sample_id[t2.split == "train"].iloc[:3].tolist() + ex[:1] + [t2.sample_id[t2.split == "test"].iloc[0]]
        up = os.path.join(d, "unused.csv"); pd.DataFrame({"sample_id": pick}).to_csv(up, index=False)
        out3 = os.path.join(d, "features_unused.csv")
        main(["--dir", tr, "--test-dir", te, "--cities", args.cities, "--drop-radius", "50", "--unused-ids", up, "--out", out3])
        t3 = pd.read_csv(out3, usecols=["sample_id", "split"]).set_index("sample_id").split
        assert (t3[pick[:3]] == "unused").all() and t3[ex[0]] == "excluded" and t3[pick[-1]] == "test", t3[pick].to_dict()
        assert t3.value_counts().to_dict() == {"train": 210 - 2 - 3, "test": 70, "excluded": 2, "unused": 3}, t3.value_counts().to_dict()
        print(f"SELFTEST PASSED: {n_expected} features, label-derived columns dropped, rows aligned, "
              f"split = {vc}; coordinate rule flags {ex}")

Merge the real exports (writes `data/features_full_1872_split.csv`; to also drop the 50 m neighbours use the shipped `unused_existing_ids.csv` as below):

In [ ]:
main()